# Práctica SQL sobre customers

**Autor:** Daniel Guzmán  
**Fecha:** 2026-04-23  
**Entorno:** Databricks

Markdown: investigación de JOINs en PySpark

## Investigación: JOINs en PySpark

En PySpark existen varios tipos de JOIN, entre los más usados:

- `inner`: devuelve solo los registros que coinciden en ambas tablas.
- `left`: devuelve todos los registros de la tabla izquierda y los coincidentes de la derecha.
- `right`: devuelve todos los registros de la tabla derecha y los coincidentes de la izquierda.
- `full`: devuelve todos los registros de ambas tablas, coincidan o no.

### Sintaxis básica

```python
df_resultado = df_a.join(df_b, df_a["id"] == df_b["id"], how="left")


Diferencias frente a SQL tradicional
En SQL tradicional se escribe la consulta como texto (SELECT ... FROM ... JOIN ...).
En PySpark se puede trabajar con la API de DataFrames usando .join().
En PySpark se suele indicar explícitamente la condición del join con columnas de cada DataFrame.
PySpark permite combinar tanto enfoque DataFrame como enfoque SQL con createOrReplaceTempView() y spark.sql().

En resumen, PySpark ofrece la misma lógica de joins que SQL, pero adaptada a operaciones sobre DataFrames distribuidos.

In [0]:
catalog = "workspace"
schema = "default"
volume = "customers_files_daniel"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print(path_volume)

In [0]:
import re

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{path_volume}/customers.csv")
)

def normalize_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name).strip("_")
    return col_name

for old_col in df.columns:
    df = df.withColumnRenamed(old_col, normalize_column(old_col))

print(df.columns)
display(df.limit(5))

In [0]:
df.createOrReplaceTempView("customers")

spark.sql("DROP TABLE IF EXISTS workspace.default.customers_delta")

spark.sql("""
CREATE TABLE workspace.default.customers_delta
USING DELTA
AS
SELECT * FROM customers
""")

In [0]:
spark.sql("SELECT * FROM customers LIMIT 5").show()

In [0]:
result_1 = spark.sql("""
    SELECT *
    FROM customers
    WHERE lower(country) = 'mexico'
""")

display(result_1)

In [0]:
result_2 = spark.sql("""
    SELECT country, COUNT(*) AS total_clientes
    FROM customers
    GROUP BY country
    ORDER BY total_clientes DESC
""")

display(result_2)

In [0]:
result_3 = spark.sql("""
    SELECT country, COUNT(*) AS total_clientes
    FROM customers
    GROUP BY country
    HAVING COUNT(*) > 5
    ORDER BY total_clientes DESC
""")

display(result_3)

In [0]:
result_4 = spark.sql("""
    SELECT country, COUNT(*) AS total_clientes
    FROM customers
    GROUP BY country
    ORDER BY total_clientes DESC
    LIMIT 5
""")

display(result_4)

In [0]:
result_5 = spark.sql("""
    SELECT CONCAT_WS(' ', first_name, last_name) AS nombre_completo,
           COUNT(*) AS frecuencia
    FROM customers
    GROUP BY CONCAT_WS(' ', first_name, last_name)
    ORDER BY frecuencia DESC
    LIMIT 1
""")

display(result_5)

In [0]:
result_6 = spark.sql("""
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY customer_id) AS ranking
    FROM customers
""")

display(result_6)

In [0]:
cliente_objetivo = spark.sql("""
    SELECT customer_id
    FROM workspace.default.customers_delta
    LIMIT 1
""").collect()[0][0]

print("Cliente objetivo:", cliente_objetivo)

In [0]:
spark.sql(f"""
    UPDATE workspace.default.customers_delta
    SET country = 'Colombia'
    WHERE customer_id = '{cliente_objetivo}'
""")

In [0]:
result_7 = spark.sql(f"""
    SELECT customer_id, country
    FROM workspace.default.customers_delta
    WHERE customer_id = '{cliente_objetivo}'
""")

display(result_7)

In [0]:
spark.sql("""
    DELETE FROM workspace.default.customers_delta
    WHERE city IS NULL OR TRIM(city) = ''
""")

In [0]:
result_8 = spark.sql("""
    SELECT COUNT(*) AS clientes_sin_ciudad
    FROM workspace.default.customers_delta
    WHERE city IS NULL OR TRIM(city) = ''
""")

display(result_8)

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_registros_restantes
    FROM workspace.default.customers_delta
""").show()